# library-hiroba の AI デモ（Google Colab 用）

`ai` は、小さな言語モデルをその場で動かします。書き方は PyHiroba と Colab で同じです。

- PyHiroba … ブラウザの中で動きます（追加のインストールは要りません）
- Colab・Jupyter … `transformers` と `torch` で動きます（下のセルで入れます）

入力した文章が外部に送られることはありません。通信が起きるのはモデルを受け取るときだけです。

各セルを上から順に実行してください。

In [ ]:
%pip install -q "library-hiroba[ai]"
# PyHiroba では、このセルの実行は不要です

## `await` が要ります

`ai` のメソッドは3つとも `await` を付けて呼びます。ノートブックのセルには、そのまま `await` を書けます。

PyHiroba は GitHub Pages で配信しているため `SharedArrayBuffer` が使えず、ブラウザ側は「待つ」形にせざるを得ません。Colab 側は待つ必要がありませんが、同じコードが両方で動くことを優先して形を揃えています。

In [ ]:
from library_hiroba import ai, ui

# 選べるモデルの一覧（approxMB は、その環境での目安の通信量）
await ai.models()

In [ ]:
# モデルを読み込みます。初回だけ時間と通信量がかかります
# 軽さを優先するなら "llmjp150m"、日本語の自然さを優先するなら "qwen15"
print(await ai.load())

In [ ]:
print(await ai.ask("日本の四季について、2行で書いて"))

In [ ]:
# 長さは max_tokens で決めます（既定は 256）
print(await ai.ask("俳句を1つ作って", max_tokens=64))

## 部品と組み合わせる

答えをそのまま `print` せず、`ui` の部品に載せると教材らしくなります。

In [ ]:
answer = await ai.ask("for文とは何か、小学生にもわかるように2行で説明して", max_tokens=96)
ui.card("for文って何？", answer, footer="AI が書いた文章です。まちがいがないか確かめてみよう")

## チャットにする

`ui.form()` と `ui.chat()` を組み合わせると、1つのセルで対話ができます。
`ai.ask()` を呼ぶので `handler` は `async def` になりますが、`ui.form()` が待ってから表示します。

（この組み合わせが動くのは Colab・Jupyter です。PyHiroba は本体側の対応待ちです。）

In [ ]:
history = []

async def talk(question):
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": await ai.ask(question, max_tokens=96)})
    return ui.chat(history, names={"user": "あなた", "assistant": "AI"})

ui.form(talk, ui.field("question", label="質問", placeholder="日本で一番高い山は？"),
        title="AI に聞いてみよう", submit_label="送信", clear_on_submit=True)

---

確認ポイント:

- `await ai.load()` が「準備ができました」と表示する
- `await ai.ask(...)` が日本語の文章を返す
- チャットの入力欄に質問を入れて送信すると、吹き出しが増えていく

小さなモデルなので、答えが事実と違うことがあります。教材では「確かめる」題材として使うのが向いています。

モデルのライセンスは配布元をご確認ください（既定の Qwen2.5 は Apache-2.0）。